# Canales

**Capítulo 4 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_convolutional-neural-networks/channels.ipynb` · [Lección original](https://d2l.ai/chapter_convolutional-neural-networks/channels.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Múltiples canales de entrada y salida múltiples
<a id="sec_channels"></a>

Mientras describimos los múltiples canales que componen cada imagen (por ejemplo, las imágenes en color tienen los canales RGB estándar para indicar la cantidad de capas rojas, verdes y azules) y convolucionales para múltiples canales en [Referencia subsec_why-conv-channels](https://d2l.ai/chapter_convolutional-neural-networks/why-conv.html#subsec-why-conv-channels), hasta ahora simplificamos todos nuestros ejemplos numéricos trabajando con una sola entrada y un solo canal de salida. Esto nos permitió pensar en nuestras entradas, núcleos de convolución y salidas cada uno como tensores bidimensionales.

Cuando añadimos canales en la mezcla, nuestras entradas y representaciones ocultas se convierten en tensores tridimensionales. Por ejemplo, cada imagen de entrada RGB tiene forma $3\times h\times w$. Nos referimos a este eje, con un tamaño de 3, como la dimensión *canal*. La noción de canales es tan antigua como las propias CNN: por ejemplo, LeNet-5 [LeCun.Jackel.Bottou.ea.1995](https://d2l.ai/chapter_references/zreferences.html) las usa. En esta sección, vamos a echar un vistazo más profundo a los núcleos de convolución con múltiples canales de entrada y salida múltiples.


In [ ]:
import torch
from laboratorio import d2l

## Múltiples canales de entrada
Cuando los datos de entrada contienen varios canales, necesitamos construir un núcleo de convolución con el mismo número de canales de entrada que los datos de entrada, para que pueda realizar una correlación cruzada con los datos de entrada. Suponiendo que el número de canales para los datos de entrada es $c_\textrm{i}$, el número de canales de entrada del núcleo de convolución también necesita ser $c_\textrm{i}$. Si la forma de ventana de nuestro núcleo de convolución es $k_\textrm{h}\times k_\textrm{w}$, entonces, cuando $c_\textrm{i}=1$, podemos pensar en nuestro núcleo de convolución como un tensor bidimensional de forma $k_\textrm{h}\times k_\textrm{w}$.

Sin embargo, cuando $c_\textrm{i}>1$, necesitamos un núcleo que contenga un tensor de forma $k_\textrm{h}\times k_\textrm{w}$ para *cada canal de entrada*. Concatenar estos tensores $c_\textrm{i}$ juntos produce un núcleo de convolución de forma $c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$. Dado que el núcleo de entrada y convolución cada uno tiene canales $c_\textrm{i}$, podemos realizar una operación de correlación cruzada en el tensor bidimensional de la entrada y el tensor bidimensional del núcleo de convolución para cada canal, añadiendo los resultados $c_\textrm{i}$ juntos (resumiendo sobre los canales) para producir un tensor bidimensional. Este es el resultado de una correlación cruzada bidimensional entre una entrada multicanal y un núcleo de convolución multicanal.

[Referencia fig_conv_multi_in](https://d2l.ai/chapter_convolutional-neural-networks/channels.html#fig-conv-multi-in) proporciona un ejemplo 
de una correlación cruzada bidimensional con dos canales de entrada. Las porciones sombreadas son el primer elemento de salida, así como los elementos tensores de entrada y núcleo utilizados para el cálculo de salida: $(1\times1+2\times2+4\times3+5\times4)+(0\times0+1\times1+3\times2+4\times3)=56$.

![Cálculo de correlación cruzada con dos canales de entrada.](../recursos/originales/conv-multi-in.svg)
<a id="fig_conv_multi_in"></a>

Para asegurarnos de que realmente entendemos lo que está pasando aquí, podemos ** implementar operaciones de correlación cruzada con múltiples canales de entrada** nosotros mismos. Observe que todo lo que estamos haciendo es realizar una operación de correlación cruzada por canal y luego sumando los resultados.


In [ ]:
def corr2d_multi_in(X, K):
    # Iterar a través de la 0a dimensión (canal) de K primero, luego añadirlos
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

Podemos construir el tensor de entrada `X` y el tensor de núcleo `K` correspondientes a los valores en [Referencia fig_conv_multi_in](https://d2l.ai/chapter_convolutional-neural-networks/channels.html#fig-conv-multi-in) para **validar la salida** de la operación de correlación cruzada.


In [ ]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

## Múltiples canales de salida
<a id="subsec_multi-output-channels"></a>

Independientemente del número de canales de entrada, hasta ahora siempre terminamos con un canal de salida. Sin embargo, como discutimos en [Referencia subsec_why-conv-channels](https://d2l.ai/chapter_convolutional-neural-networks/why-conv.html#subsec-why-conv-channels), resulta esencial tener múltiples canales en cada capa. En las arquitecturas de red neuronal más populares, en realidad aumentamos la dimensión del canal a medida que profundizamos en la red neuronal, normalmente bajando la muestra para cambiar la resolución espacial para mayor *profundidad del canal*. Intuitivamente, se podría pensar que cada canal responde a un conjunto diferente de características. La realidad es un poco más complicada que esto. Una interpretación ingenua sugeriría que las representaciones se aprenden independientemente por píxel o por canal. En cambio, los canales se optimizan para ser de utilidad conjunta. Esto significa que en lugar de asignar un solo canal a un detector de borde, puede significar simplemente que alguna dirección en el espacio del canal corresponde a detectar bordes.

Denotar por $c_\textrm{i}$ y $c_\textrm{o}$ el número de canales de entrada y salida, respectivamente, y por $k_\textrm{h}$ y $k_\textrm{w}$ la altura y anchura del núcleo. Para obtener una salida con múltiples canales, podemos crear un tensor de núcleo de forma $c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$ para *cada* canal de salida. Los concatenamos en la dimensión del canal de salida, de modo que la forma del núcleo de convolución es $c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$. En las operaciones de correlación cruzada, el resultado de cada canal de salida se calcula a partir del núcleo de convolución correspondiente a ese canal de salida y toma la entrada de todos los canales en el tensor de entrada.

Implementamos una función de correlación cruzada para **calcular la salida de múltiples canales** como se muestra a continuación.


In [ ]:
def corr2d_multi_in_out(X, K):
    # Iterar a través de la 0a dimensión de K, y cada vez, realizar
    # operaciones de correlación cruzada con entrada X. Todos los resultados son
    # Apilados juntos
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

Construimos un núcleo de convolución trivial con tres canales de salida concatenando el tensor del núcleo para `K` con `K+1` y `K+2`.


### Nota docente de Hespérides

Comprueba primero las formas y el supuesto arquitectónico: localidad, compartición de pesos o conexión residual. El explorador permite seguir ventana, multiplicaciones y suma. En PyTorch, Conv2d implementa correlación cruzada; en aprendizaje profundo se suele llamar convolución a esta operación. El autoencoder 91 amplía el patrón MLP con reconstrucción; VAE se trata como contraste conceptual, al no existir un original válido en las fuentes locales.

Vínculo con los apuntes: sesión 4, «Canales».


In [ ]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

A continuación, realizamos operaciones de correlación cruzada en el tensor de entrada `X` con el tensor de núcleo `K`. Ahora la salida contiene tres canales. El resultado del primer canal es consistente con el resultado del tensor de entrada `X` anterior y el canal de entrada múltiple, núcleo de canal de salida única.


In [ ]:
corr2d_multi_in_out(X, K)

## $1\times 1$ Capa Convolucional
<a id="subsec_1x1"></a>

Al principio, una convolución **$1 \times 1$**, es decir, $k_\textrm{h} = k_\textrm{w} = 1$, no parece tener mucho sentido. Después de todo, una convolución correlaciona píxeles adyacentes. Una convolución $1 \times 1$ obviamente no. Sin embargo, son operaciones populares que a veces se incluyen en los diseños de redes profundas complejas [Lin.Chen.Yan.2013,Szegedy.Ioffe.Vanhoucke.ea.2017](https://d2l.ai/chapter_references/zreferences.html). Veamos con cierto detalle lo que realmente hace.

Debido a que se utiliza la ventana mínima, la convolución $1\times 1$ pierde la capacidad de las capas convolucionales más grandes para reconocer patrones consistentes en interacciones entre elementos adyacentes en las dimensiones de altura y anchura. El único cálculo de la convolución $1\times 1$ ocurre en la dimensión del canal.

[Referencia fig_conv_1x1](https://d2l.ai/chapter_convolutional-neural-networks/channels.html#fig-conv-1x1) muestra el cálculo de correlación cruzada
con el núcleo de convolución $1\times 1$ con 3 canales de entrada y 2 canales de salida. Tenga en cuenta que las entradas y salidas tienen la misma altura y anchura. Cada elemento en la salida se deriva de una combinación lineal de elementos * en la misma posición* en la imagen de entrada. Se podría pensar que la capa convolucional $1\times 1$ constituye una capa totalmente conectada aplicada en cada ubicación de píxel para transformar los valores de entrada correspondientes $c_\textrm{i}$ en valores de salida $c_\textrm{o}$. Debido a que ésta sigue siendo una capa convolucional, los pesos se atan a través de la ubicación de píxel. Así, la capa convolucional $1\times 1$ requiere pesos $c_\textrm{o}\times c_\textrm{i}$ (más el sesgo). También tenga en cuenta que las capas convolucionales son típicamente seguidas por no linealidades. Esto asegura que las convoluciones $1 \times 1$ no pueden plegarse simplemente en otras convoluciones.

![Correlación cruzada con filtro $1\times 1$, tres canales de entrada y dos de salida. Se conservan altura y anchura.](../recursos/originales/conv-1x1.svg)
<a id="fig_conv_1x1"></a>

Vamos a comprobar si esto funciona en la práctica: implementamos una convolución $1 \times 1$ usando una capa totalmente conectada. Lo único es que necesitamos hacer algunos ajustes a la forma de los datos antes y después de la multiplicación de la matriz.


In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    # Multiplicación de matrices en la capa totalmente conectada
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

Al realizar convoluciones $1\times 1$, la función anterior es equivalente a la función de correlación cruzada previamente implementada `corr2d_multi_in_out`. Comprobemos esto con algunos datos de muestra.


In [ ]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

## Discusión
Los canales nos permiten combinar lo mejor de ambos mundos: MLPs que permiten no linealidades significativas y convoluciones que permiten el análisis *localizado* de características. En particular, los canales permiten a la CNN razonar con múltiples características, como detectores de bordes y formas al mismo tiempo. También ofrecen un intercambio práctico entre la drástica reducción de parámetros derivada de la invarianza de la traducción y la localidad, y la necesidad de modelos expresivos y diversos en la visión informática.

Nota, sin embargo, que esta flexibilidad viene a un precio. Dada una imagen de tamaño $(h \times w)$, el costo para calcular una convolución $k \times k$ es $\mathcal{O}(h \cdot w \cdot k^2)$. Para los canales de entrada y salida $c_\textrm{i}$ y $c_\textrm{o}$, respectivamente, esto aumenta a $\mathcal{O}(h \cdot w \cdot k^2 \cdot c_\textrm{i} \cdot c_\textrm{o})$. Para una imagen de píxel $256 \times 256$ con un núcleo $5 \times 5$ y canales de entrada y salida $128$, respectivamente, esto asciende a más de 53 mil millones de operaciones (contamos multiplicaciones y adiciones por separado). Más adelante nos encontraremos con estrategias eficaces para reducir el costo, por ejemplo, al requerir que las operaciones en canal sean bloques-diagonales, lo que conduce a arquitecturas como ResNeXt [Xie.Girshick.Dollar.ea.2017](https://d2l.ai/chapter_references/zreferences.html).

## Ejercicios
1. Supongamos que tenemos dos núcleos de convolución de tamaño $k_1$ y $k_2$, respectivamente (sin no linealidad en el medio).
    1. Demostrar que el resultado de la operación puede ser expresado por una sola convolución.
    1. ¿Cuál es la dimensión de la convolución única equivalente?
    1. ¿Es verdad lo contrario, es decir, siempre puedes descomponer una convolución en dos más pequeñas?
1. Suponga una entrada de forma $c_\textrm{i}\times h\times w$ y un núcleo de convolución de forma $c_\textrm{o}\times c_\textrm{i}\times k_\textrm{h}\times k_\textrm{w}$, relleno de $(p_\textrm{h}, p_\textrm{w})$ y zancada de $(s_\textrm{h}, s_\textrm{w})$.
    1. ¿Cuál es el costo computacional (multiplicaciones y adiciones) para la propagación hacia delante?
    1. ¿Cuál es la huella de la memoria?
    1. ¿Cuál es la huella de memoria para el cálculo atrasado?
    1. ¿Cuál es el costo computacional de la retropropagación?
1. ¿Por qué factor aumenta el número de cálculos si duplicamos el número de canales de entrada $c_\textrm{i}$ y el número de canales de salida $c_\textrm{o}$? ¿Qué pasa si duplicamos el relleno?
1. ¿Son exactamente iguales las variables `Y1` y `Y2` en el ejemplo final de esta sección? ¿Por qué?
1. Exprese las convoluciones como una multiplicación de matriz, incluso cuando la ventana de convolución no es $1 \times 1$.
1. Su tarea es implementar convoluciones rápidas con un núcleo $k \times k$. Uno de los candidatos del algoritmo es escanear horizontalmente a través de la fuente, leyendo una tira de ancho $k$ y computando la tira de salida de ancho $1$ un valor a la vez. La alternativa es leer una tira de ancho $k + \Delta$ y calcular una tira de salida de ancho $\Delta$. ¿Por qué es preferible esta última? ¿Hay un límite a lo grande que debe elegir $\Delta$?
1. Supongamos que tenemos una matriz $c \times c$.
    1. ¿Cuánto más rápido es multiplicarse con una matriz bloque-diagonal si la matriz se divide en bloques $b$?
    1. ¿Cuál es la desventaja de tener $b$ bloques? ¿Cómo se puede arreglar, al menos en parte?


[Debate del original](https://discuss.d2l.ai/t/70)
